In [34]:
from pathlib import Path
import mlflow
import mlflow.pytorch
from torchvision.models import convnext_tiny
import timm
import torch
import torch.nn as nn

# LOAD classifier model
tracking_uri = Path("../experiments/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_uri}")
# WEIGHTS_PATH = "convext-tiny-7626a94ac30f40bb8b88d3c7e69b9eae.pt"
# WEIGHTS_PATH = "swinv2tiny-7f572b99e4224dbe98221ffff3557390.pt"
WEIGHTS_PATH = "swinv2tiny-defaultsampl-2ec89cf7ed2e4bac921a69f5902285ab.pt"
# run_id = "feb27708e1d34a1d834936e0c3d8d6a2" # swintiny
# run_id = "7f572b99e4224dbe98221ffff3557390" # swinv2tiny from blueberries.mergedpinkpurple
run_id = "2ec89cf7ed2e4bac921a69f5902285ab" # swinv2tiny default sampling from blueberries.mergedpinkpurple
# run_id = "7626a94ac30f40bb8b88d3c7e69b9eae" # HPO convNext
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)
# artifacts = client.list_artifacts(run_id)
# artifacts = client.download_artifacts(run_id, "", ".")
# print(run.data.params['data']) # this is string -> i have to use regular expr to parse mean/std -> ugly!
classifier = mlflow.pytorch.load_model(f"runs:/{run_id}/best_model")
# type(classifier), classifier
# print(artifacts)

print(classifier.head.fc.weight)
# print(classifier.classifier[2].weight.data) # for convnext
torch.save(classifier.state_dict(), WEIGHTS_PATH)

# model = timm.create_model("swinv2_tiny_window8_256", pretrained=False, num_classes=5, img_size=64)
type(classifier)

Parameter containing:
tensor([[ 0.0015, -0.0036, -0.0019,  ..., -0.0026,  0.0089, -0.0076],
        [ 0.0044,  0.0039,  0.0031,  ..., -0.0063,  0.0134, -0.0099],
        [ 0.0050, -0.0058,  0.0049,  ...,  0.0043,  0.0084, -0.0002],
        [-0.0017, -0.0045, -0.0063,  ...,  0.0013,  0.0067, -0.0068],
        [ 0.0043,  0.0071,  0.0013,  ..., -0.0052,  0.0020, -0.0053]],
       device='cuda:0', requires_grad=True)


timm.models.swin_transformer_v2.SwinTransformerV2

In [35]:
model = timm.create_model("swinv2_tiny_window8_256", pretrained=False, num_classes=5, img_size=64)
model.load_state_dict(torch.load(WEIGHTS_PATH))
# model.eval()
with torch.inference_mode():
  print(model.head.fc.weight)

Parameter containing:
tensor([[ 0.0015, -0.0036, -0.0019,  ..., -0.0026,  0.0089, -0.0076],
        [ 0.0044,  0.0039,  0.0031,  ..., -0.0063,  0.0134, -0.0099],
        [ 0.0050, -0.0058,  0.0049,  ...,  0.0043,  0.0084, -0.0002],
        [-0.0017, -0.0045, -0.0063,  ...,  0.0013,  0.0067, -0.0068],
        [ 0.0043,  0.0071,  0.0013,  ..., -0.0052,  0.0020, -0.0053]],
       requires_grad=True)


In [24]:
model = convnext_tiny()
model.classifier[2] = nn.Linear(model.classifier[2].in_features, 5)
# print(model.classifier[2].weight.data)
model.load_state_dict(torch.load(WEIGHTS_PATH))
with torch.inference_mode():
  print(model.classifier[2].weight.data)

RuntimeError: Error(s) in loading state_dict for ConvNeXt:
	Missing key(s) in state_dict: "features.0.0.weight", "features.0.0.bias", "features.0.1.weight", "features.0.1.bias", "features.1.0.layer_scale", "features.1.0.block.0.weight", "features.1.0.block.0.bias", "features.1.0.block.2.weight", "features.1.0.block.2.bias", "features.1.0.block.3.weight", "features.1.0.block.3.bias", "features.1.0.block.5.weight", "features.1.0.block.5.bias", "features.1.1.layer_scale", "features.1.1.block.0.weight", "features.1.1.block.0.bias", "features.1.1.block.2.weight", "features.1.1.block.2.bias", "features.1.1.block.3.weight", "features.1.1.block.3.bias", "features.1.1.block.5.weight", "features.1.1.block.5.bias", "features.1.2.layer_scale", "features.1.2.block.0.weight", "features.1.2.block.0.bias", "features.1.2.block.2.weight", "features.1.2.block.2.bias", "features.1.2.block.3.weight", "features.1.2.block.3.bias", "features.1.2.block.5.weight", "features.1.2.block.5.bias", "features.2.0.weig

In [ ]:
def print_artifacts(client, run_id, path=""):
    for item in client.list_artifacts(run_id, path):
        print(item.path)
        if item.is_dir:
            print_artifacts(client, run_id, item.path)

# print_artifacts(client, run_id)
client.list_artifacts(run_id, "best_model")
client.download_artifacts(run_id, "best_model/data")

In [ ]:
## Production -> How to run it in ML dashboard
from torchvision.models import convnext_tiny
import torchvision.transforms.v2 as v2
import torch
import torch.nn as nn
N_CLASSES = 5
DEVICE = torch.device("cuda:0")

# # CNN model
# model = convnext_tiny()
# model.classifier[2] = nn.Linear(model.classifier[2].in_features, N_CLASSES)
# model.load_state_dict(torch.load(WEIGHTS_PATH))
# 
# mean = [0.4291, 0.5388, 0.3654]
# std =  [0.1871, 0.2131, 0.1977] # modified due to interpolationMode change
# 
# inference_transform = v2.Compose([
#   v2.ToImage(),
#   v2.Resize((64,64), interpolation=v2.InterpolationMode.BICUBIC), # modified interpolationMode from default
#   v2.ToDtype(torch.float32, scale=True),
#   v2.Normalize(mean=mean, std=std) # depends on which model i choose , what dataset it was trained
# ])
# 
# crop_transformed = torch.randn(2,3,64,64).to(DEVICE)
# crop_transformed = inference_transform(crop).unsqueeze(0).to(DEVICE)
#  with torch.no_grad():
#     logits = classifier(crop_transformed).cpu()
#     classifier_probs = nn.Softmax(dim=1)(logits).cpu() # (1, n_classes)

# transformer model
model = timm.create_model("swinv2_tiny_window8_256", pretrained=False, num_classes=N_CLASSES, img_size=64)
model.load_state_dict(torch.load(WEIGHTS_PATH))
model.eval()
with torch.inference_mode():
  print(model.head.fc.weight)

mean = [0.4291, 0.5388, 0.3654]
std =  [0.1871, 0.2131, 0.1977] # modified due to interpolationMode change

inference_transform = v2.Compose([
  v2.ToImage(),
  v2.Resize((64,64), interpolation=v2.InterpolationMode.BICUBIC), # modified interpolationMode from default
  v2.ToDtype(torch.float32, scale=True),
  v2.Normalize(mean=mean, std=std) # depends on which model i choose , what dataset it was trained
])

crop_transformed = torch.randn(2,3,64,64).to(DEVICE)
# crop_transformed = inference_transform(crop).unsqueeze(0).to(DEVICE)
with torch.no_grad():
  logits = classifier(crop_transformed).cpu()
  classifier_probs = nn.Softmax(dim=1)(logits).cpu() # (1, n_classes)

print(logits.shape)


Parameter containing:
tensor([[ 0.0018,  0.0014,  0.0082,  ..., -0.0045, -0.0005, -0.0091],
        [-0.0047,  0.0089,  0.0056,  ..., -0.0084,  0.0002, -0.0107],
        [-0.0072,  0.0137,  0.0103,  ..., -0.0097,  0.0013, -0.0149],
        [ 0.0147,  0.0014, -0.0025,  ..., -0.0098,  0.0082, -0.0103],
        [ 0.0009,  0.0095,  0.0193,  ...,  0.0044, -0.0009, -0.0107]],
       requires_grad=True)
torch.Size([2, 5])
